# 대청댐 조류경보 예측 모델 학습

이 노트북은 현재 정리된 모델 입력 데이터셋을 사용해 조류경보 예측 모델을 학습, 평가, 저장하는 실행 문서입니다.

현재 기본 입력은 트리 기반 Gradient Boosting용 데이터입니다.

```text
src/data/processed/model_input/tree_gradient_boosting/algae_tree_station_expanded.csv
```

이 파일은 `ALGAE_DATA.csv`에 아래 모델링 편의 컬럼을 추가한 버전입니다.

- `split`: date-level chronological holdout 기준 train/valid 구분
- `target_alert_next`: `next_log_cells >= log10(1000 + 1)`로 만든 분류 target

구현 코드는 노트북 안에 길게 두지 않고 아래 두 파일로 분리했습니다.

```text
src/config/model_config.py
src/model_pipeline.py
```

## 1. 역할 분리

현재 단계에서는 과한 모듈화 대신 아래 3개 책임만 분리합니다.

| 위치 | 역할 |
| --- | --- |
| `src/config/model_config.py` | 데이터 경로, 컬럼명, target, threshold, 모델 후보, artifact 경로 설정 |
| `src/model_pipeline.py` | 데이터 로드, feature 선택, split, 학습, 평가, 예측, 저장 함수 |
| `model_training.ipynb` | 실행 순서, 결과 확인, 성능표/중요 변수 확인 |

이 노트북은 실험 실행과 결과 확인용이고, 재사용 가능한 로직은 `src/model_pipeline.py`에 둡니다.

## 2. Import

In [ ]:
from __future__ import annotations

import importlib
from pathlib import Path

import pandas as pd

from src.config import model_config as config
from src import model_pipeline as pipeline

# 개발 중 모듈을 수정한 뒤 노트북에서 다시 실행할 수 있게 reload합니다.
importlib.reload(config)
importlib.reload(pipeline)

## 3. Config 확인

현재 기본 설정은 트리 기반 Gradient Boosting용 입력 데이터입니다. 비트리 스케일 데이터로 실험하려면 `src/config/model_config.py`에서 `MODEL_INPUT_PATH`를 `NON_TREE_MODEL_INPUT_PATH`로 바꾸면 됩니다.

In [ ]:
print("MODEL_INPUT_PATH:", config.MODEL_INPUT_PATH)
print("DATE_COLUMN:", config.DATE_COLUMN)
print("ID_COLUMNS:", config.ID_COLUMNS)
print("REGRESSION_TARGET:", config.REGRESSION_TARGET)
print("CLASSIFICATION_TARGET:", config.CLASSIFICATION_TARGET)
print("REGRESSION_TARGET_TRANSFORM:", config.REGRESSION_TARGET_TRANSFORM)
print("REGRESSION_MODEL_CANDIDATES:", config.REGRESSION_MODEL_CANDIDATES)
print("CLASSIFICATION_MODEL_CANDIDATES:", config.CLASSIFICATION_MODEL_CANDIDATES)

## 4. 데이터 로드 및 구조 확인

`ALGAE_DATA.csv`는 `date x loc_encoded x station` 구조입니다. 같은 수질 조사 행이 station 4개 기상값과 각각 결합되어 반복됩니다.

In [ ]:
df = pipeline.load_model_input()
print("shape:", df.shape)
print("missing:", int(df.isna().sum().sum()))
print("duplicates:", int(df.duplicated().sum()))
print("date range:", df[config.DATE_COLUMN].min(), "~", df[config.DATE_COLUMN].max())
print("split counts:")
print(df[config.SPLIT_COLUMN].value_counts())
print("station counts:")
print(df[config.STATION_COLUMN].value_counts().sort_index())
print("loc_encoded counts:")
print(df[config.SITE_COLUMN].value_counts().sort_index())
df.head()

## 5. Train / Validation Split

현재 split은 **date-level chronological holdout** 방식입니다.

- 과거 날짜 블록: train
- 미래 날짜 블록: valid

이 방식은 시계열 예측에서 `look-ahead bias`, `temporal leakage`, station-expanded 구조의 `group leakage`를 줄이기 위한 설정입니다.

In [ ]:
train_df, valid_df = pipeline.time_based_train_valid_split(df)
print("train:", train_df.shape, train_df[config.DATE_COLUMN].min(), "~", train_df[config.DATE_COLUMN].max())
print("valid:", valid_df.shape, valid_df[config.DATE_COLUMN].min(), "~", valid_df[config.DATE_COLUMN].max())
print("train alert rate:", train_df[config.CLASSIFICATION_TARGET].mean())
print("valid alert rate:", valid_df[config.CLASSIFICATION_TARGET].mean())

## 6. Feature Column 확인

아래 컬럼은 feature에서 제외합니다.

- `date`
- `split`
- `next_log_cells`
- `target_alert_next`

`log_target`은 현재 시점 세포수 로그값이므로 feature로 사용할 수 있습니다. 단, `cyano_cells`와 정보가 중복될 수 있어 추후 ablation 실험 대상입니다.

In [ ]:
feature_columns = pipeline.get_feature_columns(train_df)
print("feature count:", len(feature_columns))
print(feature_columns)

## 7. 모델 학습

기본 후보는 Gradient Boosting 계열입니다.

- LightGBM
- XGBoost
- HistGradientBoosting

설치되어 있지 않은 외부 모델은 자동으로 건너뜁니다.

In [ ]:
trained = pipeline.train_candidate_models(train_df, feature_columns)
print("regression models:", list(trained["regression_models"].keys()))
print("classification models:", list(trained["classification_models"].keys()))

## 8. 평가 및 Best Model 선택

회귀는 `RMSE` 최소 기준, 분류는 조류경보 미탐 방지를 위해 `Recall` 최대 기준으로 best model을 선택합니다.

In [ ]:
metrics_df, threshold_df = pipeline.evaluate_candidate_models(trained, valid_df)
best_models = pipeline.select_best_models(metrics_df)
print("best_models:", best_models)
metrics_df

In [ ]:
threshold_df.sort_values(["model_name", "threshold"]).head(30)

## 9. 예측 결과와 Feature Importance

In [ ]:
regression_predictions, classification_predictions = pipeline.make_predictions(trained, valid_df, best_models)
feature_importance = pipeline.get_feature_importance(trained, valid_df, best_models)

print("regression_predictions:", regression_predictions.shape)
print("classification_predictions:", classification_predictions.shape)
print("feature_importance:", feature_importance.shape)
feature_importance.head(20)

In [ ]:
regression_predictions.head()

In [ ]:
classification_predictions.head()

## 10. Artifact 저장

아래 셀을 실행하면 모델, 지표, threshold 후보, 예측 결과, feature importance가 `artifacts/` 아래에 저장됩니다.

In [ ]:
pipeline.save_artifacts(
    trained=trained,
    metrics_df=metrics_df,
    threshold_df=threshold_df,
    best_models=best_models,
    regression_predictions=regression_predictions,
    classification_predictions=classification_predictions,
    feature_importance=feature_importance,
)

print("saved model dir:", config.OUTPUT_DIR)
print("saved metric dir:", config.METRIC_DIR)
print("saved prediction dir:", config.PREDICTION_DIR)
print("saved explain dir:", config.EXPLAIN_DIR)

## 11. 한 번에 실행하기

위 셀들을 단계별로 실행해도 되고, 아래 함수 하나로 전체 파이프라인을 실행할 수도 있습니다.

```python
result = pipeline.run_training_pipeline(save=True)
```

노트북에서 디버깅할 때는 위의 단계별 셀을 권장합니다.

In [ ]:
# result = pipeline.run_training_pipeline(save=True)